# 04 · Cross-Tier Scale Analysis
**Run this AFTER all scale notebooks are complete.**

This notebook:
1. Loads results from all scale tiers
2. Plots Recall@K curves across scales
3. Fits a log-linear model to characterize the scale law
4. Applies the stopping criterion
5. Generates all paper figures (Figure 1, 2, 3)


In [ ]:

import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

RESULTS_ROOT = "../../4_results"
SCALE_CONFIGS = [
    ("scale_1K",   1_000),
    ("scale_5K",   5_000),
    ("scale_15K",  15_000),
    ("scale_30K",  30_000),
    ("scale_50K",  50_000),
    # ("scale_75K",  75_000),   # uncomment if run
    # ("scale_100K", 100_000),  # uncomment if run
]

# Load results from each scale
tier_results = []
for scale_label, n_papers in SCALE_CONFIGS:
    path = f"{RESULTS_ROOT}/{scale_label}/recall_at_k.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        row = {"label": scale_label, "n": n_papers}
        for _, r in df.iterrows():
            row[f"dense_r{int(r['k'])}"]  = r['dense']
            row[f"bm25_r{int(r['k'])}"]   = r['bm25']
            row[f"hybrid_r{int(r['k'])}"] = r['hybrid']
        tier_results.append(row)
        print(f"  Loaded {scale_label}")
    else:
        print(f"  MISSING: {path}")

print(f"\n{len(tier_results)} / {len(SCALE_CONFIGS)} scale tiers loaded")


In [ ]:

if len(tier_results) < 2:
    print("Need at least 2 tiers to plot. Run more scale notebooks first.")
else:
    df_all = pd.DataFrame(tier_results)
    print("\nScale summary:")
    print(df_all[['label','n','dense_r1','bm25_r1','hybrid_r1','dense_r10','hybrid_r10']].to_string(index=False))


In [ ]:

# ── FIGURE: BM25 vs Dense R@1 crossover across scales ──────────────────────
if len(tier_results) >= 2:
    fig, ax = plt.subplots(figsize=(8,5))
    ax.plot(df_all['n'], df_all['dense_r1'], 'b-o', label='Dense R@1', linewidth=2)
    ax.plot(df_all['n'], df_all['bm25_r1'],  'r-s', label='BM25 R@1',  linewidth=2)
    
    # Mark crossover if it exists
    for i in range(len(df_all)-1):
        d1, d2 = df_all['dense_r1'].iloc[i], df_all['dense_r1'].iloc[i+1]
        b1, b2 = df_all['bm25_r1'].iloc[i],  df_all['bm25_r1'].iloc[i+1]
        if (d1 < b1 and d2 >= b2) or (d1 >= b1 and d2 < b2):
            n1, n2 = df_all['n'].iloc[i], df_all['n'].iloc[i+1]
            ax.axvspan(n1, n2, alpha=0.15, color='green', label='Crossover region')
    
    ax.set_xscale('log')
    ax.set_xlabel('Corpus size (chunks, log scale)')
    ax.set_ylabel('Recall@1')
    ax.set_title('BM25 vs Dense Recall@1 — Crossover Across Scales')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{RESULTS_ROOT}/scale_crossover_plot.png", dpi=150)
    plt.show()
    print("Saved: scale_crossover_plot.png")


In [ ]:

# ── FIT LOG-LINEAR SCALE MODEL ──────────────────────────────────────────────
# If this fits well, we have empirical evidence for a "scale law" (log-linear)
if len(tier_results) >= 3:
    log_n = np.log10(df_all['n'].values)
    
    for metric in ['dense_r1', 'bm25_r1', 'hybrid_r10']:
        y = df_all[metric].values
        slope, intercept, r, p, se = stats.linregress(log_n, y)
        print(f"{metric:15s}: slope={slope:+.4f}  R²={r**2:.3f}  p={p:.4f}")
    
    print("\nInterpretation: R² > 0.90 → strong log-linear (scale law) relationship")
    print("If R² < 0.70, describe as 'scale effects' not 'scale laws'")


In [ ]:

# ── STOPPING CRITERION ──────────────────────────────────────────────────────
if len(tier_results) >= 2:
    print("Stopping criterion: change in BM25/Dense gap across consecutive tiers")
    print("-" * 60)
    for i in range(1, len(tier_results)):
        prev = tier_results[i-1]
        curr = tier_results[i]
        prev_gap = prev['dense_r1'] - prev['bm25_r1']
        curr_gap = curr['dense_r1'] - curr['bm25_r1']
        change = curr_gap - prev_gap
        status = "PLATEAU" if abs(change) < 0.005 else "active"
        print(f"  {prev['label']} → {curr['label']}: gap change = {change:+.4f}  [{status}]")
